# Test evaluation on Colab

Runs the test evaluation (`docs/cloud.md`) with one script. This notebook only drives
`scripts/run_test_eval.py` and holds no logic of its own.

**Before you start**
1. *Runtime → Change runtime type → GPU: **A100**.* (L4 works too, slower; not the T4.)
2. The `MISTRAL_API_KEY` secret must exist (🔑 icon in
   the left sidebar, *Notebook access* on).
3. Every run must be finished, with its full validation: the fine-tuning runs
   (`notebooks/colab_finetune.ipynb`) and the reference runs
   (`scripts/run_reference.py`, `scripts/run_api_reference.py`).

The test split is scored **once** per system, and the final model is named on validation
before that. A disconnect is harmless: reconnect, run cells 2, 3 and 4 again, and it
continues where it stopped.

## 1 · Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → L4 or A100"
name = torch.cuda.get_device_name(0)
memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {memory_gb:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}")
assert torch.cuda.is_bf16_supported() and memory_gb > 20, "Need an L4 or A100, not a T4"

## 2 · Mount Drive, get the code, install

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
url = "https://github.com/HFossdal/voxtral-norwegian.git"
if os.path.exists("/content/voxtral-norwegian"):
    !git -C /content/voxtral-norwegian pull -q {url} main
else:
    !git clone -q {url} /content/voxtral-norwegian
%cd /content/voxtral-norwegian
!git log --oneline -1
!pip install -q -e .
# Colab's preinstalled torchvision is built for its own torch version, not the
# pinned one, and can crash on import. This project doesn't use it.
!pip uninstall -y -q torchvision
# Same for torchao: peft refuses to add LoRA layers when an old torchao is
# installed (Colab ships 0.10; peft 0.21 needs >= 0.16). Not used here either.
!pip uninstall -y -q torchao
# pip then warns about other preinstalled Colab packages (cudf, diffusers, ...)
# wanting other versions of pandas, numba or pyarrow. They are not used here.

## 3 · Restart the runtime

The install replaces Colab's own torch and transformers with the project's pinned
versions. They are only used after a restart. This cell restarts the runtime on
purpose: Colab will report that the session crashed. **That is expected.** Then go on
with cell 4. You don't need to run cells 1–2 again.

The red `ERROR: pip's dependency resolver ...` lines above are warnings about
Colab packages this project never imports. The install itself succeeded if
it printed `Building editable for voxnor ... done`.

In [ ]:
import os

os.kill(os.getpid(), 9)

## 4 · Score every system on test

Three parts, announced in the log with `=== n/3 ... ===`:
1. find every system's validation run and name the final model: the fine-tuned
   candidate with the lowest full-validation WER (`test_plan.json`);
2. score each system on the test split (about 3–4 hours in all; the Mistral API part
   costs about $2);
3. write the test reports and a zip.

In [ ]:
import os

from google.colab import userdata

# Read from Colab's secret store, never written into the notebook.
os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
%cd /content/voxtral-norwegian
out = "/content/drive/MyDrive/voxtral-runs"
!python scripts/run_test_eval.py --config configs/cloud.yaml --output-root {out}

## 5 · Look at the results

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

root = Path("/content/drive/MyDrive/voxtral-runs")
print((root / "test_plan.json").read_text())
display(Markdown((root / "reports" / "final_test.md").read_text()))

Everything worth keeping is also in `MyDrive/voxtral-runs/final_test_results.zip`.
Download it from Drive, or run the cell below.

In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/voxtral-runs/final_test_results.zip")